In [1]:
from qubic.lib.Qhdf5 import HDF5Dict
import matplotlib.pyplot as plt
import healpy as hp
import numpy as np
from glob import glob
import os
from PIL import Image
import io
import ctypes
ctypes.CDLL("/pbs/software/redhat-9-x86_64/mpich/4.2.1/lib/libmpi.so.12", mode=ctypes.RTLD_GLOBAL)
from qubic.lib.MapMaking.Qmap_plotter import plot_cross_spectrum
from IPython.display import Image as IPyImage
from qubic.lib.QskySim import get_angular_profile
from copy import deepcopy

In [2]:
hdf = HDF5Dict()
DATA_PATH = "parametric_d0_DB_test_nu_ref_150"

# Check Number of Realisation & Spectra

In [3]:
realisation_path = sorted(glob(os.path.join(DATA_PATH, "Dict", "*.h5")))
spectra_path = realisation_path = sorted(glob(os.path.join(DATA_PATH, "Spectrum", "*.h5")))

print("Number of realisation : ", len(realisation_path))
print("Number of Spectra : ", len(spectra_path))
print("Missing Spectra : ", len(realisation_path) - len(spectra_path))

Number of realisation :  166
Number of Spectra :  166
Missing Spectra :  0


# Check Parameters

In [6]:
def remove_seed(p):
    p = deepcopy(p)  # avoid modifying original dict
    try:
        del p["QUBIC"]["NOISE"]["seed_noise"]
        del p["PLANCK"]["seed_noise"]
    except KeyError:
        pass
    return p

In [7]:
ref_simu_params = remove_seed(hdf.load_dict(realisation_path[-1])["parameters"])
ref_spectrum_params = remove_seed(hdf.load_dict(spectra_path[0])["parameters"])

print(ref_simu_params)

{'save_iter': 1, 'foldername': 'sigma_r_cmb_dust_nsub_6', 'filename': 'test', 'lastite': False, 'CMB': {'cmb': True, 'seed': 1, 'r': 0, 'Alens': 1}, 'Foregrounds': {'fit_components': True, 'fit_mixing_matrix': True, 'bin_mixing_matrix': 6, 'blind_method': 'minimize', 'Dust': {'Dust_in': True, 'Dust_out': True, 'type': 'parametric', 'model': 'd0', 'beta_init': [1.54, 0.02, 10000000], 'nside_beta_in': 0, 'nside_beta_out': 0, 'nu0': 353.0, 'l_corr': 10000000, 'amplification': 1}, 'Synchrotron': {'Synchrotron_in': False, 'Synchrotron_out': False, 'type': 'parametric', 'model': 's0', 'beta_init': [-3, 0.01], 'nu0': 150, 'amplification': 1}, 'CO': {'CO_in': False, 'CO_out': False, 'nu0': 230.538, 'polarization_fraction': 0.01}}, 'QUBIC': {'instrument': 'DB', 'configuration': 'FI', 'npointings': 5000, 'nsub_in': 40, 'nsub_out': 18, 'convolution_in': True, 'convolution_out': False, 'fwhm_rec': None, 'use_reference_pcg': True, 'preconditioner': False, 'NOISE': {'ndet': 1, 'npho150': 1, 'npho220

In [8]:

for i, real in enumerate(realisation_path):
    d = remove_seed(hdf.load_dict(real)["parameters"])
    test_all_same = (d == ref_simu_params)
    if not test_all_same:
        print(f"Realisation {i} does not have the same parameters than reference one.")
    
for i, spec in enumerate(spectra_path):
    d = remove_seed(hdf.load_dict(spec)["parameters"])
    test_all_same = (d == ref_spectrum_params)
    if not test_all_same:
        print(f"Spectra {i} does not have the same parameters than reference one.")
        
print("If no print above, simulation and spectra parameters are the same across all realisation ! :)")

Realisation 0 does not have the same parameters than reference one.
Realisation 1 does not have the same parameters than reference one.
Realisation 2 does not have the same parameters than reference one.
Realisation 3 does not have the same parameters than reference one.
Realisation 4 does not have the same parameters than reference one.
Realisation 5 does not have the same parameters than reference one.
Realisation 6 does not have the same parameters than reference one.
Realisation 7 does not have the same parameters than reference one.
Realisation 8 does not have the same parameters than reference one.
Realisation 9 does not have the same parameters than reference one.
Realisation 10 does not have the same parameters than reference one.
Realisation 11 does not have the same parameters than reference one.
Realisation 12 does not have the same parameters than reference one.
Realisation 13 does not have the same parameters than reference one.
Realisation 14 does not have the same parame

In [ ]:
stop

NameError: name 'stop' is not defined

# Check X-Spectra

In [ ]:
frames = []
plt.ioff()
for i, file_name in enumerate(sorted(glob(os.path.join(DATA_PATH, "Spectrum", "*.h5")))):
    dict = hdf.load_dict(file_name)
    nus = dict["nus"]
    ell = dict["ell"]
    Dls = dict["Dls"]
    Nls = dict["Nls"]
    Dls_noiseless = Dls - Nls
    N = nus.size - 7

    fig = plot_cross_spectrum(
        nus=nus[:N],
        ell=ell,
        Dl=Dls[:N, :N],
        Dl_err=Nls[:N, :N],
        ymodel=None,
        mode="Dl",
        nrec=N,
        figsize=(12, 10),
        title=f" (QUBIC only) - ID {i}",
    )
    
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    frames.append(Image.open(buf))
    plt.close(fig)

# Save to an in-memory bytes buffer
gif_buffer = io.BytesIO()
frames[0].save(gif_buffer,
                save_all=True,
                append_images=frames[1:],
                duration=100,
                loop=0,
                optimize=True,
                format='GIF')
gif_buffer.seek(0)

# Return an IPython Image object (will be displayed automatically)
IPyImage(data=gif_buffer.getvalue())

# Check Noise profiles

In [ ]:
plt.ioff()
frames = []
Nrec = 2

for i, file_name in enumerate(sorted(glob(os.path.join(DATA_PATH, "Dict", "*.h5")))):
    maps_res = hdf.load_item(file_name, "maps_in_convolved") - hdf.load_item(file_name, "maps")[:Nrec]

    fig = plt.figure(figsize=(12, 8))

    for inu in range(Nrec):
        plt.subplot(1, Nrec, inu + 1)

        get_angular_profile(
            maps_res[inu],
            doplot=True,
            allstokes=True,
            nbins=80,
            thmax=20
        )

        plt.title(f"{nus[inu].round(1)} GHz - ID {i}")

    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    frames.append(Image.open(buf).copy())  # évite bug mémoire
    buf.close()

    plt.close(fig)
    
gif_buffer = io.BytesIO()
frames[0].save(gif_buffer,
                save_all=True,
                append_images=frames[1:],
                duration=50,
                loop=0,
                optimize=True,
                format='GIF')
gif_buffer.seek(0)

# Return an IPython Image object (will be displayed automatically)
IPyImage(data=gif_buffer.getvalue())